# Lab 3 — Change and uncertainty

**Twenty-five minutes.**

Plot a measured trajectory inside the uncertainty band created by the
method. Then use measured storms to ask whether a single table value and
a rainfall proxy describe what the watershed actually did.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import json
import os
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cnkit import CN_from_PQ, fit_asymptotic

USE_EARTH_ENGINE = False
GAGE = "01646000"
LAT, LON = 38.97594, -77.24581
YEARS = [2001, 2004, 2007, 2010, 2013, 2016, 2019]


## 1. The trajectory everyone can analyze

These are recorded outputs from a successful live Earth Engine run over
Difficult Run. The line is measured land-cover change. The band is the
poor-to-good hydrologic-condition assumption over the same pixels.


In [ ]:
trajectory = pd.read_csv(PREPARED_DIR / "difficult_run_gee_trajectory.csv").set_index("year")
change = float(trajectory.fair.iloc[-1] - trajectory.fair.iloc[0])
mean_spread = float(trajectory.spread.mean())
ratio = mean_spread / abs(change)

print("2001 CN                         %.4f" % trajectory.fair.iloc[0])
print("2019 CN                         %.4f" % trajectory.fair.iloc[-1])
print("measured change                %+.4f CN" % change)
print("mean condition spread           %.4f CN" % mean_spread)
print("assumption / signal ratio        %.1f" % ratio)


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.fill_between(
    trajectory.index,
    trajectory.good,
    trajectory.poor,
    color="#c85d45",
    alpha=0.22,
    label="poor-to-good condition assumption",
)
ax.plot(
    trajectory.index,
    trajectory.fair,
    "o-",
    color="#007f92",
    lw=2.6,
    label="measured land-cover trajectory",
)
ax.set(xlabel="year", ylabel="composite curve number")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")
plt.show()


## 2. Ask the gage instead of the table

`fit_asymptotic` uses measured rainfall and runoff events. It does not
assume the table value is correct.


In [ ]:
fits = []
for watershed, gage, table_cn in [
    ("Difficult Run", "01646000", 75.5),
    ("Accotink Creek", "01654000", 77.9),
]:
    events = pd.read_csv(DATA_DIR / ("events_" + gage + ".csv"))
    fit20 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.20)
    fit05 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.05)
    fits.append(
        {
            "watershed": watershed,
            "events": len(events),
            "table_CN": table_cn,
            "fitted_CN_lambda_020": fit20.cn_inf,
            "r2_lambda_020": fit20.r2,
            "fitted_CN_lambda_005": fit05.cn_inf,
            "r2_lambda_005": fit05.r2,
        }
    )
pd.DataFrame(fits).set_index("watershed").round(3)


For Difficult Run, the table is high by about 5.8 CN units. Accotink
differs in the opposite direction and its fitted relation is weak. The
table error is not a simple correction that transfers between basins.


## 3. Antecedent condition from an observed state

Split the Difficult Run events by root-zone wetness on the day before
each event. Compare the dry and wet thirds without inventing new AMC
classes.


In [ ]:
events = pd.read_csv(DATA_DIR / "events_01646000.csv", parse_dates=["start"])
moisture = pd.read_csv(
    DATA_DIR / "soilmoisture_power_01646000.csv", parse_dates=["date"]
).set_index("date")

events["previous_day"] = events.start.dt.normalize() - pd.Timedelta(days=1)
events["root_zone_wetness"] = events.previous_day.map(moisture.GWETROOT)
events["event_cn"] = CN_from_PQ(events.P_in.values, events.Q_in.values)
valid = events.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["root_zone_wetness", "event_cn"]
)
valid["wetness_third"] = pd.qcut(
    valid.root_zone_wetness, 3, labels=["driest", "middle", "wettest"]
)
antecedent = valid.groupby("wetness_third", observed=True).agg(
    events=("event_cn", "size"),
    median_wetness=("root_zone_wetness", "median"),
    median_event_cn=("event_cn", "median"),
    median_runoff_ratio=("runoff_ratio", "median"),
)
antecedent.round(3)


## 4. Earth Engine trajectory — a selected watershed

This application uses seven years and returns the hydrologic-condition
band with the trajectory. Set `USE_EARTH_ENGINE = True` to run it for the
selected watershed.


In [ ]:
live_trajectory = None

if USE_EARTH_ENGINE:
    activate_full_cnkit()
    import ee
    from cnkit.delineate import watershed_from_gage, watershed_from_point
    from cnkit.gee import initialise
    from cnkit.workflows import cn_trajectory

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass("Earth Engine project ID: ")
    ee.Authenticate()
    initialise(project=project)
    watershed = watershed_from_gage(GAGE) if GAGE else watershed_from_point(LAT, LON)

    live_trajectory = cn_trajectory(
        watershed=watershed,
        project=project,
        years=YEARS,
        condition="fair",
        soils="sda",
        progress=lambda done, total, year: print("%d/%d  %d" % (done, total, year)),
    )
    print(live_trajectory)
    print(live_trajectory.summary())
else:
    print("Reference trajectory active.")
    print("Set USE_EARTH_ENGINE=True to calculate a selected watershed trajectory.")


## Final reporting statement

Write six lines:

1. Lambda and curve-number calibration used.
2. Land-cover source, vintage, and resolution.
3. Soil source and unmapped fraction.
4. Hydrologic condition and its poor-to-good sensitivity.
5. Composite convention: distributed runoff, weighted CN, or weighted S.
6. Measured trajectory beside the uncertainty band.

**Source anchors:** USGS NWIS; ACIS/PRISM; NASA POWER; Annual NLCD;
Woodward et al. (2003); NEH-630 Chapters 9 and 10.
